# 19 - Hyperparameter Tuning

> 📘 **Instructor Curriculum**

### The story

Think about making tea.

Before you start, **you** choose a few things:

- how much sugar
- how long to boil it
- how high the flame is

Then the tea makes itself. You only chose the settings.

A model works the same way.

```
   YOU choose these before training   ->  hyperparameters
        max_depth = 4
        criterion = 'gini'

   The MODEL learns these while       ->  parameters
   training
        the question inside each box
        the number it splits on
        the answer in each leaf
```

You already do this in Java. In `application.yml` you set the thread pool size
and the timeout. You do not write the objects the app creates at runtime.
Same wall here.

Now the important part:

**`fit()` cannot choose `max_depth` for you.**

Why not? Because you only find out if depth 4 was a good idea *after* the tree
is built and tested. By then `fit()` is already finished.

So someone has to build many trees and compare them. This notebook does that in
three ways:

1. by hand - Step 2
2. try every option - Step 3
3. try a few random options - Step 6

Cells 1 to 4 are the same iris setup as notebook 18. A table, then `x` and `y`,
then 120 rows to learn from and 30 rows kept aside for the final test.

In [1]:
import pandas as pd
from sklearn.datasets import load_iris
iris = load_iris()
table = pd.DataFrame(iris.data,columns=iris.feature_names)
table.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


In [2]:
table['target'] = iris.target
table.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


In [3]:
x = table.drop(['target'], axis='columns')
y = iris.target  

In [4]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size = 0.20, random_state = 0)

## Step 1 - The three settings you will change

A decision tree has many settings. Your notebook uses three.

| setting | values you try | what it does |
| --- | --- | --- |
| `criterion` | `'gini'`, `'entropy'` | how it measures *"how mixed is this box?"* |
| `splitter` | `'best'`, `'random'` | how hard it looks for a good cut point |
| `max_depth` | `1` to `6` | how many questions the tree may ask |

**`criterion`**

Both words mean the same thing: *how mixed is this group?*

- `entropy` is the formula you did by hand in notebook 16.
- `gini` is a shorter formula: `1 - sum of (p × p)`.

Both give 0 for a clean box. Both give a big number for a mixed box. Gini has
no log in it, so it is faster. That is why it is the default. Most of the time
the two build the same tree.

**`splitter`**

The tree has to pick a cut point, like *"petal length <= 2.5"*.

- `'best'` checks every possible cut and keeps the best one. Slow and careful.
- `'random'` picks one random cut per column, then keeps the best of those few.
  Fast and lucky.

**`max_depth`**

This is how many questions the tree may ask before it must give an answer.

```
   max_depth = 1            max_depth = 3
        ?                        ?
       / \                      / \
      A   B                    ?   ?
                              / \ / \
                             ?  ? ?  ?

   only 2 answers            up to 8 answers
   too few to name           enough room to memorise
   3 flower types            single flowers
```

Too small, and the tree learns almost nothing.
Too big, and the tree just remembers the training rows.

There is no formula for the right value. You have to try. That is why this
notebook exists.

In [5]:
from sklearn.tree import DecisionTreeClassifier

## Step 2 - First way: pick four by hand

Here **you** are the search. You choose four sets of settings yourself.

```
         criterion   splitter   max_depth
   dt1    gini        best         2
   dt2    entropy     best         5
   dt3    gini        random       4
   dt4    entropy     random       6
```

Four trees. Same data. Only the settings are different.

In [6]:
dt1 = DecisionTreeClassifier(criterion= 'gini', splitter= 'best', max_depth=2)
dt2 = DecisionTreeClassifier(criterion= 'entropy', splitter= 'best', max_depth=5)
dt3 = DecisionTreeClassifier(criterion= 'gini', splitter= 'random', max_depth= 4)
dt4 = DecisionTreeClassifier(criterion= 'entropy', splitter= 'random', max_depth= 6)

All four trees are trained here.

A Jupyter cell only prints its last line, so you only see `dt4` in the output.
The other three are trained too.

In [7]:
dt1.fit(X_train, y_train)
dt2.fit(X_train, y_train)
dt3.fit(X_train, y_train)
dt4.fit(X_train, y_train)

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'entropy'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'random'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",6
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamp

In [8]:
print('dt1 -> ',dt1.score(X_test, y_test))
print('dt2 -> ',dt2.score(X_test, y_test))
print('dt3 -> ',dt3.score(X_test, y_test))
print('dt4 -> ',dt4.score(X_test, y_test))

dt1 ->  0.9666666666666667
dt2 ->  1.0
dt3 ->  0.9
dt4 ->  1.0


Put the settings and the scores side by side:

```
   dt1   gini    / best   / depth 2   ->  0.967
   dt2   entropy / best   / depth 5   ->  1.000
   dt3   gini    / random / depth 4   ->  0.900   <- worst
   dt4   entropy / random / depth 6   ->  1.000
```

This looks fine. It is not. There are three problems.

**Problem 1. Two models are tied at 1.000.**
Which one do you ship? You cannot say.

**Problem 2. 1.000 only means 30 out of 30.**
The test set has 30 rows. So one flower is worth 3.3%. With so few rows, 1.000
and 0.967 are almost the same thing.

**Problem 3. This is the big one.**
You looked at the test set four times, then picked a winner from it. The test
set was your final exam paper. You were not supposed to open it early. Now it
has helped you choose, so it can no longer tell you the truth.

Two more things:

- You tried only 4 of the 24 possible sets.
- `splitter='random'` uses luck, and nothing here fixes the luck. Run the cell
  again and these four numbers move. The numbers above are from the saved run.

Remember `dt3`, the worst one. It comes back in Step 5.

## Step 3 - Second way: let the computer try all of them

First count how many sets there are:

```
   2 criterion  x  2 splitter  x  6 depths  =  24 sets of settings
```

`GridSearchCV` builds all 24 for you. *Grid* means the full table of options.
*CV* means cross validation, and that is Step 4.

Two lines set it up.

**`dt = DecisionTreeClassifier()`**

This is an empty tree. Think of it as a blank form. `GridSearchCV` takes 24
copies of this form and fills each copy with one set of settings. Your `dt`
itself is never trained.

**`options = {...}`**

A dictionary. The key is the name of the setting. The value is the list of
things to try.

```python
{'criterion': ('gini','entropy'),      # try these 2
 'splitter' : ('best','random'),       # try these 2
 'max_depth': list(range(1,7))}        # try 1, 2, 3, 4, 5, 6
```

Two rules for this dictionary:

- The key must be spelled exactly like the real setting. `'criteria'` gives an
  error, not a warning.
- The values can be a tuple, a list or a range. It does not matter which.

`list(range(1,7))` gives `[1,2,3,4,5,6]`. `range` stops before the last number,
so you write 7 when you want 6.

In [9]:
dt = DecisionTreeClassifier()

In [10]:
from sklearn.model_selection import GridSearchCV

In [12]:
options = {'criterion': ('gini','entropy'),'splitter': ('best','random'),'max_depth' : list(range(1,7)) }

## Step 4 - What that one `fit()` really does

This is the part that was not explained in class. It is the whole point.

`gs.fit(X_train, y_train)` **never sees `X_test`.** You did not give the test
set to it, so it cannot cheat.

Then how does it mark 24 models without a test set?

It makes its own small test sets, out of the training data.

It cuts the 120 training rows into 5 equal parts. Each part has 24 rows. Then
it trains 5 times. Each time, a different part is kept back for marking.

```
        120 training rows, cut into 5 parts of 24

        part   1     2     3     4     5
             +-----+-----+-----+-----+-----+
   round 1   | MARK|train|train|train|train|  -> score
   round 2   |train| MARK|train|train|train|  -> score
   round 3   |train|train| MARK|train|train|  -> score
   round 4   |train|train|train| MARK|train|  -> score
   round 5   |train|train|train|train| MARK|  -> score
             +-----+-----+-----+-----+-----+

              average of those 5 = mean_test_score
```

So every set of settings is marked on rows it did not learn from. Five times.
Then the five marks are averaged.

One average of 5 marks is much safer than one mark on 30 rows.

Now count the work:

```
   24 sets  x  5 rounds  =  120 trees, from that one fit() call
```

That is why the cell takes a moment to finish.

(Cross validation on its own is notebook 11.)

### The settings of GridSearchCV

| setting | you wrote | default | what it does |
| --- | --- | --- | --- |
| `estimator` | `dt` | - | the blank model to copy |
| `param_grid` | `options` | - | the dictionary of settings to try |
| `cv` | nothing | `5` | how many parts to cut the training rows into |
| `scoring` | nothing | the model's own score, so accuracy | what *better* means |
| `n_jobs` | nothing | `1` | put `-1` to use all your CPU cores |
| `refit` | nothing | `True` | after the search, train the winner again on all 120 rows |
| `verbose` | nothing | `0` | `1` to `3` prints progress while it works |

`refit=True` is why `gs.best_estimator_` exists, and why `gs.predict(X_test)`
works straight away. The winner is already trained and ready.

The text printed under the cell is only sklearn repeating your settings back to
you.

In [13]:
gs = GridSearchCV(dt,options)
gs.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",DecisionTreeClassifier()
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'criterion': ('gini', ...), 'max_depth': [1, 2, ...], 'splitter': ('best', ...)}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",None
,"verbose verbose: int, default=0Controls the verbo

## Step 5 - Reading the result book

`cv_results_` is a dictionary. Every list inside it has 24 items. Item number 3
in one list belongs with item number 3 in all the others.

`cv_results_['params']` is the list of the 24 sets it tried.

```
   criterion   max_depth   splitter      index
     gini    ->    1    ->  best           0
     gini    ->    1    ->  random         1
     gini    ->    2    ->  best           2
     gini    ->    2    ->  random         3
      ...                                 ...
   entropy   ->    6    ->  random        23
```

The order is not random. sklearn sorts your keys A to Z - `criterion`,
`max_depth`, `splitter` - and then changes the **last one fastest**.

It works like a clock. Seconds move fastest, hours move slowest.

- `splitter` changes on every row
- `max_depth` changes every 2 rows
- `criterion` changes once, in the middle

Other useful keys in the same dictionary:

- `mean_test_score` - the average mark of each set
- `rank_test_score` - 1 for the best, 2 for second, and so on
- `split0_test_score` … `split4_test_score` - the 5 single marks behind each average
- `mean_fit_time` - how long each one took

In [14]:
gs.cv_results_['params']

[{'criterion': 'gini', 'max_depth': 1, 'splitter': 'best'},
 {'criterion': 'gini', 'max_depth': 1, 'splitter': 'random'},
 {'criterion': 'gini', 'max_depth': 2, 'splitter': 'best'},
 {'criterion': 'gini', 'max_depth': 2, 'splitter': 'random'},
 {'criterion': 'gini', 'max_depth': 3, 'splitter': 'best'},
 {'criterion': 'gini', 'max_depth': 3, 'splitter': 'random'},
 {'criterion': 'gini', 'max_depth': 4, 'splitter': 'best'},
 {'criterion': 'gini', 'max_depth': 4, 'splitter': 'random'},
 {'criterion': 'gini', 'max_depth': 5, 'splitter': 'best'},
 {'criterion': 'gini', 'max_depth': 5, 'splitter': 'random'},
 {'criterion': 'gini', 'max_depth': 6, 'splitter': 'best'},
 {'criterion': 'gini', 'max_depth': 6, 'splitter': 'random'},
 {'criterion': 'entropy', 'max_depth': 1, 'splitter': 'best'},
 {'criterion': 'entropy', 'max_depth': 1, 'splitter': 'random'},
 {'criterion': 'entropy', 'max_depth': 2, 'splitter': 'best'},
 {'criterion': 'entropy', 'max_depth': 2, 'splitter': 'random'},
 {'criterion

`best_params_` gives you the winner as a ready dictionary:

```
   {'criterion': 'gini', 'max_depth': 4, 'splitter': 'random'}
```

Now look back at Step 2 and find `dt3`.

**gini, random, depth 4. It is the same tree.**

In Step 2 it scored 0.900 and came last of your four. Here it comes first.

Nothing about the tree changed. Only the way of marking changed. One small
test of 30 rows called it the worst. Five fair rounds called it the best.

This is the best reason in the notebook to stop picking settings by hand.

Two more things you can ask for:

- `gs.best_score_` - the winner's average mark
- `gs.best_estimator_` - the winner, already trained on all 120 rows

In [15]:
gs.best_params_

{'criterion': 'gini', 'max_depth': 4, 'splitter': 'random'}

`mean_test_score` is one number for each set of settings. It is the average of
that set's 5 marks. It is in the same order as `params`.

⚠️ Be careful with the word **test** here.

It does **not** mean `X_test`. It means the part that was kept back inside the
training data. Your real test set is still unopened.

In [16]:
scores=gs.cv_results_['mean_test_score']
scores

array([0.69166667, 0.69166667, 0.95      , 0.76666667, 0.93333333,
       0.95833333, 0.925     , 0.975     , 0.93333333, 0.95      ,
       0.925     , 0.925     , 0.69166667, 0.68333333, 0.93333333,
       0.9       , 0.93333333, 0.90833333, 0.93333333, 0.86666667,
       0.93333333, 0.925     , 0.925     , 0.925     ])

`np.argmax(scores)` gives you the **place** of the biggest number. Not the
number itself.

```
   small example
   scores    = [0.60, 0.90, 0.70]
   argmax    ->  1        the place
   scores[1] ->  0.90     the number
```

Your run gives `7`. And `params[7]` is gini / depth 4 / random.

That is exactly what `best_params_` already said. So you just found the winner
by hand. Good - it shows there is nothing magic inside.

In [17]:
import numpy as np
np.argmax(scores)

np.int64(7)

In [18]:
scores[2]

np.float64(0.95)

One small mistake to fix here.

`scores[2]` is set number 2 (gini / depth 2 / `'best'`), which is 0.95. That is
not the winner. `argmax` said **7**.

```python
scores[np.argmax(scores)]   # 0.975  <- the best score
gs.best_score_              # the same number, ready made
```

### See all 24 in one picture

The search fills a table of options, so let us draw that table.

Dark green is a good score. Pale is a bad score.

In [ ]:
# every combination GridSearchCV tried, drawn as the grid it really is
import numpy as np
import matplotlib.pyplot as plt

params = gs.cv_results_['params']
scores = gs.cv_results_['mean_test_score']

depths = sorted({p['max_depth'] for p in params})
rows = [(c, s) for c in ('gini', 'entropy') for s in ('best', 'random')]
grid = np.full((len(rows), len(depths)), np.nan)
for p, sc in zip(params, scores):
    grid[rows.index((p['criterion'], p['splitter'])), depths.index(p['max_depth'])] = sc

fig, ax = plt.subplots(figsize=(8, 3.4))
im = ax.imshow(grid, cmap='YlGn')
ax.set_xticks(range(len(depths)), [f"depth {d}" for d in depths])
ax.set_yticks(range(len(rows)), [f"{c} / {s}" for c, s in rows])
for r in range(grid.shape[0]):
    for c in range(grid.shape[1]):
        ax.text(c, r, f"{grid[r, c]:.3f}", ha='center', va='center', fontsize=9)

best = np.unravel_index(np.nanargmax(grid), grid.shape)
ax.add_patch(plt.Rectangle((best[1] - .5, best[0] - .5), 1, 1, fill=False, edgecolor='red', lw=2.5))
ax.set_title("mean_test_score of all 24 combinations  (red box = best_params_)")
plt.colorbar(im, label='mean CV accuracy')
plt.show()

Three things the picture shows that the plain numbers hide.

**1. The `depth 1` column is pale.**
One question gives only two answers. But iris has three flower types. So a
depth 1 tree can never get the third one right. It is stuck near 2 out of 3.
That column is not a bad choice. It is an impossible one.

**2. From depth 2, the picture is flat.**
Once the tree can ask two questions, the job is nearly done. The rest of the
table is small ups and downs around the same value.

**3. The red box is only just ahead.**
Each part holds 24 flowers, so one flower moves an average by about 0.008.
First place and third place are often one or two flowers apart. So read
`best_params_` as *"a good setting"*, not *"the only setting"*.

## Step 6 - Third way: try only a few, picked at random

Grid search has one problem: multiplication.

```
   3 settings, small lists         ->     24 sets     (this notebook)
   6 settings with 5 values each   ->   3000 sets
   3000 sets x 5 rounds            ->  15000 trees for one fit()
```

You cannot sit and wait for that.

`RandomizedSearchCV` says: do not try all of them. Pick a few at random and try
only those.

Two things change in the syntax. Everybody gets caught by these.

| | GridSearchCV | RandomizedSearchCV |
| --- | --- | --- |
| the dictionary argument is called | `param_grid` | **`param_distributions`** |
| the values can be | lists only | lists **or** a range like `randint(1, 20)` |
| how many it runs | all of them | `n_iter` of them |

The second row is the real reason this class exists. A grid can only try the
values you typed. A range can reach `max_depth = 17` without you typing 17.

Your two lines:

```python
samples = 5
rs = RandomizedSearchCV(dt, param_distributions=options, n_iter=samples)
```

`n_iter=5` means: pick 5 of the 24 and try only those.

```
   5 sets x 5 rounds = 25 trees        (grid search did 120)
```

Cells 19 and 20 build `dt` and `options` again with the same values. Nothing
new. `dt` from Step 3 was never trained, so this is only a clean start.

In [19]:
from sklearn.model_selection import RandomizedSearchCV

In [20]:
dt = DecisionTreeClassifier()

In [21]:
options = {'criterion': ('gini','entropy'),'splitter': ('best','random'),'max_depth' : list(range(1,7)) }

In [22]:
samples = 5  
rs = RandomizedSearchCV(dt, param_distributions=options, n_iter=samples)

In [23]:
rs.fit(X_train, y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",DecisionTreeClassifier()
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'criterion': ('gini', ...), 'max_depth': [1, 2, ...], 'splitter': ('best', ...)}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",5
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` fo

In [24]:
rs.cv_results_['params']

[{'splitter': 'best', 'max_depth': 2, 'criterion': 'gini'},
 {'splitter': 'best', 'max_depth': 5, 'criterion': 'gini'},
 {'splitter': 'best', 'max_depth': 5, 'criterion': 'entropy'},
 {'splitter': 'random', 'max_depth': 1, 'criterion': 'entropy'},
 {'splitter': 'random', 'max_depth': 4, 'criterion': 'gini'}]

Five sets of settings, picked at random out of the 24:

```
   splitter / depth / criterion        mean_test_score
   best   /  2  / gini                     0.950   <- best of the five
   best   /  5  / gini                     0.925
   best   /  5  / entropy                  0.925
   random /  1  / entropy                  0.667   <- the depth 1 problem again
   random /  4  / gini                     0.917
```

`np.argmax` gives 0, and `best_params_` gives the first one. They agree.

The keys print in a different order than in grid search. That means nothing.
Random search does not sort them.

### The trade

```
   GridSearchCV        120 trees   ->   0.975
   RandomizedSearchCV   25 trees   ->   0.950
```

You did 79% less work, and lost 2.5 points.

For 24 sets that is a bad deal. Just run the grid.
For 3000 sets it is the only deal you have.

The normal habit: random search first, to find the good area. Then a small grid
inside that area.

If you raise `n_iter`, random search slowly turns into grid search. At
`n_iter=24` they are the same thing, only slower.

In [25]:
rs.best_params_

{'splitter': 'best', 'max_depth': 2, 'criterion': 'gini'}

In [26]:
scores = rs.cv_results_['mean_test_score']
scores

array([0.95      , 0.925     , 0.925     , 0.66666667, 0.91666667])

In [27]:
import numpy as np
np.argmax(scores)

np.int64(0)

## The syntax, in one place

The two searches are really the same object. Everything below works for both.

```python
search.fit(X_train, y_train)   # runs sets x rounds trees. never sees X_test
search.best_params_            # the winning settings, as a dictionary
search.best_score_             # the winner's average mark
search.best_estimator_         # the winner, already trained on all training rows
search.cv_results_             # the full result book
search.predict(X_test)         # works, because refit=True by default
```

Only three things are different:

```
                      GridSearchCV      RandomizedSearchCV
   class name         GridSearchCV      RandomizedSearchCV
   dictionary arg     param_grid        param_distributions
   how many           all of them       n_iter of them
```

### One warning about every number in this notebook

Nothing here sets `random_state`.

But `splitter='random'` uses luck, and `RandomizedSearchCV` picks its 5 sets by
luck. So **run it again and the winner changes.**

When you want two runs to match, fix the luck in both places:

```python
dt = DecisionTreeClassifier(random_state=0)
rs = RandomizedSearchCV(dt, options, n_iter=5, random_state=0)
```

**If you remember only one thing:** you choose the settings, `fit()` learns the
rest, and a search is only a loop that tries your settings in a fair way. It
marks each one on parts kept back from the training rows, so the real test set
stays closed until the end.